In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# imblearn's Pipeline, not sklearn's, to ensure SMOTE only applies to training data
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_excel('../data/raw/E Commerce Dataset.xlsx', sheet_name='E Comm')
df = df.drop('CustomerID', axis=1, errors='ignore') # Drop ID

X = df.drop('Churn', axis=1)
y = df['Churn']

# Split the data (Stratified ensures the 16% churn rate is maintained in both sets)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

Training set: 4504 samples
Testing set: 1126 samples


In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'str']).columns

# Numeric Pipeline: Impute missing with median -> Standardize
numeric_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Impute missing with mode -> One-Hot Encode
categorical_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessor built successfully.")

Preprocessor built successfully.


In [ ]:
#Logistic Regression
pipeline_lr = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Train the pipeline
pipeline_lr.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [5]:
# Cell 5: Evaluate the baseline model on the test set
# The pipeline automatically applies the preprocessing steps (without SMOTE) to X_test
y_pred = pipeline_lr.predict(X_test)
y_proba = pipeline_lr.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Baseline Results ===")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

=== Logistic Regression Baseline Results ===

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.79      0.86       936
           1       0.44      0.83      0.58       190

    accuracy                           0.79      1126
   macro avg       0.70      0.81      0.72      1126
weighted avg       0.87      0.79      0.82      1126


ROC-AUC Score: 0.8838

Confusion Matrix:
[[737 199]
 [ 32 158]]
